In [6]:
import requests, re

In [2]:
# Step 1: find current infrastructure versions
resp = requests.get(
    "https://www.trassenfinder.de/api/web/infrastrukturen",
    headers={"Referer": "https://trassenfinder.de/"}
)

print("Status:", resp.status_code)
print("Body:", resp.text[:2000])

Status: 200
Body: [{"id":8,"anzeigename":"Jahresfahrplan 2026","fahrplanjahr":2026,"gueltig_von":"2025-12-14","gueltig_bis":"2026-12-12"},{"id":3,"anzeigename":"S-Bahn Berlin 2026","fahrplanjahr":2026,"gueltig_von":"2025-12-14","gueltig_bis":"2026-12-12"}]


In [3]:
session = requests.Session()
session.headers.update({"Referer": "https://trassenfinder.de/"})

In [4]:
candidates = [
    "https://www.trassenfinder.de/api/web/infrastrukturen/8/betriebsstellen?limit=3",
    "https://www.trassenfinder.de/api/web/infrastrukturen/8/zugkonfigurationen",
    "https://www.trassenfinder.de/api/web/infrastrukturen/8/lokomotiven",
    "https://www.trassenfinder.de/api/web/infrastrukturen/8/fahrzeuge",
    "https://www.trassenfinder.de/api/web/zugkonfigurationen",
    "https://www.trassenfinder.de/api/web/lokomotiven",
    "https://openapi.trassenfinder.de/api-docs",
    "https://openapi.trassenfinder.de/v3/api-docs",
    "https://openapi.trassenfinder.de/swagger.json",
    "https://openapi.trassenfinder.de/openapi.json",
]

for url in candidates:
    r = session.get(url)
    print(f"{r.status_code}  {url}")
    if r.status_code == 200 and r.text:
        print("  →", r.text[:200])

404  https://www.trassenfinder.de/api/web/infrastrukturen/8/betriebsstellen?limit=3
404  https://www.trassenfinder.de/api/web/infrastrukturen/8/zugkonfigurationen
404  https://www.trassenfinder.de/api/web/infrastrukturen/8/lokomotiven
404  https://www.trassenfinder.de/api/web/infrastrukturen/8/fahrzeuge
404  https://www.trassenfinder.de/api/web/zugkonfigurationen
404  https://www.trassenfinder.de/api/web/lokomotiven
404  https://openapi.trassenfinder.de/api-docs
200  https://openapi.trassenfinder.de/v3/api-docs
  → <!doctype html>
<html lang="de">
  <head>
    <meta charset="utf-8" />
    <link href="/apple-touch-icon.png" rel="apple-touch-icon" sizes="180x180" />
    <link href="/favicon-32x32.png" rel="icon" s
200  https://openapi.trassenfinder.de/swagger.json
  → <!doctype html>
<html lang="de">
  <head>
    <meta charset="utf-8" />
    <link href="/apple-touch-icon.png" rel="apple-touch-icon" sizes="180x180" />
    <link href="/favicon-32x32.png" rel="icon" s
200  https://openapi.t

In [8]:
html = session.get("https://openapi.trassenfinder.de/").text
print("Full HTML:")
print(html)

Full HTML:
<!doctype html>
<html lang="de">
  <head>
    <meta charset="utf-8" />
    <link href="/apple-touch-icon.png" rel="apple-touch-icon" sizes="180x180" />
    <link href="/favicon-32x32.png" rel="icon" sizes="32x32" type="image/png" />
    <link href="/favicon-16x16.png" rel="icon" sizes="16x16" type="image/png" />
    <link color="#5bbad5" href="/safari-pinned-tab.svg" rel="mask-icon" />
    <meta content="#282d37" name="msapplication-TileColor" />
    <meta content="#ffffff" name="theme-color" />
    <meta content="width=device-width, initial-scale=1" name="viewport" />
    <meta content="Trassenfinder OpenAPI" name="description" />
    <link rel="manifest" href="/manifest.json" crossorigin="use-credentials" />
    <title>Trassenfinder OpenAPI</title>
    <script type="module" crossorigin src="/assets/index-DD9XDope.js"></script>
    <link rel="stylesheet" crossorigin href="/assets/index-BFZ0M9hf.css">
  </head>
  <body>
    <noscript>You need to enable JavaScript to run this

In [9]:
js = session.get("https://openapi.trassenfinder.de/assets/index-DD9XDope.js").text
print("Bundle size:", len(js), "chars")

# Find spec/docs URLs
spec_hits = re.findall(r'["\']([^"\']*(?:api-docs|swagger|openapi|spec|yaml|json)[^"\']{0,60})["\']', js, re.IGNORECASE)
print("\n--- Spec-like URLs ---")
for h in sorted(set(spec_hits)):
    print(" ", h)

# Find all /api/ paths
api_paths = re.findall(r'["\']((?:https?://[^"\']*)?/api/[^"\']{3,80})["\']', js)
print("\n--- /api/ paths ---")
for p in sorted(set(api_paths)):
    print(" ", p)

# Find any trassenfinder.de URLs
tf_urls = re.findall(r'["\'](https?://[^"\']*trassenfinder[^"\']{0,100})["\']', js)
print("\n--- trassenfinder URLs ---")
for u in sorted(set(tf_urls)):
    print(" ", u)

Bundle size: 1245217 chars

--- Spec-like URLs ---

--- /api/ paths ---

--- trassenfinder URLs ---


In [10]:
js = session.get("https://openapi.trassenfinder.de/assets/index-DD9XDope.js").text

# Try backtick template literals too, and looser patterns
all_strings = re.findall(r'["\`]([^"\`]{8,120})["\`]', js)

interesting = [s for s in all_strings if any(kw in s.lower() for kw in [
    'infrastruktur', 'suchen', 'betrieb', 'trasse', 'fahrzeug', 
    'swagger', 'openapi', 'v3/', 'yaml', '.json', '/api'
])]

print(f"Found {len(interesting)} interesting strings:")
for s in interesting:
    print(" ", s)

Found 53 interesting strings:
  /openapi
  https://trassenfinder.de/datenschutz
  https://trassenfinder.de/datenschutz
   ersetzt.\n\n### getInfrastruktur\n\n#### Response\n\n- 
  ordnungsrahmen -> betriebsstellen
  result -> *_route -> routenpunkte -> triebfahrzeug
  result -> *_route -> routenpunkte -> triebfahrzeug_key
  triebfahrzeug_key
  result -> *_route -> routenpunkte -> abweichendes_zweites_triebfahrzeug
  result -> *_route -> routenpunkte -> abweichendes_zweites_triebfahrzeug_key
  triebfahrzeug_key
  infrastrukturId
  ordnungsrahmen -> betriebsstellen -> ds100
  ordnungsrahmen -> mutter_betriebsstellen -> ds100
  ordnungsrahmen -> mutter_betriebsstellen -> tochterbetriebsstellen
   erweitert.\n\n### getInfrastrukturen\n\n#### Request\n\n- Einschr├żnkung durch 
  wegpunkte -> betriebsstelle -> ds100
  /api/v4/infrastrukturen/{infrastruktur_id}
  /api/v5/infrastrukturen/{infrastruktur_id}
  /api/v4/infrastrukturen
  /api/v5/infrastrukturen
  /api/v4/infrastrukturen/{infrastru

In [11]:
# Try the actual OpenAPI spec
spec = session.get("https://openapi.trassenfinder.de/openapi")
print("Spec status:", spec.status_code)
print("Spec content-type:", spec.headers.get("Content-Type"))
print(spec.text[:3000])

Spec status: 200
Spec content-type: text/html
<!doctype html>
<html lang="de">
  <head>
    <meta charset="utf-8" />
    <link href="/apple-touch-icon.png" rel="apple-touch-icon" sizes="180x180" />
    <link href="/favicon-32x32.png" rel="icon" sizes="32x32" type="image/png" />
    <link href="/favicon-16x16.png" rel="icon" sizes="16x16" type="image/png" />
    <link color="#5bbad5" href="/safari-pinned-tab.svg" rel="mask-icon" />
    <meta content="#282d37" name="msapplication-TileColor" />
    <meta content="#ffffff" name="theme-color" />
    <meta content="width=device-width, initial-scale=1" name="viewport" />
    <meta content="Trassenfinder OpenAPI" name="description" />
    <link rel="manifest" href="/manifest.json" crossorigin="use-credentials" />
    <title>Trassenfinder OpenAPI</title>
    <script type="module" crossorigin src="/assets/index-DD9XDope.js"></script>
    <link rel="stylesheet" crossorigin href="/assets/index-BFZ0M9hf.css">
  </head>
  <body>
    <noscript>You ne

In [12]:
for url in [
    "https://openapi.trassenfinder.de/api/v5/infrastrukturen",
    "https://openapi.trassenfinder.de/api/v5/version",
    "https://openapi.trassenfinder.de/api/v5/stammdaten",
    "https://openapi.trassenfinder.de/api/v4/infrastrukturen",
    "https://openapi.trassenfinder.de/api/v4/version",
]:
    r = session.get(url)
    print(f"\n{r.status_code}  {url}")
    if r.text:
        print(r.text[:500])


404  https://openapi.trassenfinder.de/api/v5/infrastrukturen

404  https://openapi.trassenfinder.de/api/v5/version

404  https://openapi.trassenfinder.de/api/v5/stammdaten

404  https://openapi.trassenfinder.de/api/v4/infrastrukturen

404  https://openapi.trassenfinder.de/api/v4/version


In [13]:
for url in [
    "https://www.trassenfinder.de/api/v5/infrastrukturen",
    "https://www.trassenfinder.de/api/v5/version",
    "https://www.trassenfinder.de/api/v5/stammdaten",
    "https://www.trassenfinder.de/api/v4/infrastrukturen",
    "https://www.trassenfinder.de/api/v4/version",
    "https://trassenfinder.de/api/v5/infrastrukturen",
    "https://trassenfinder.de/api/v5/version",
]:
    r = session.get(url)
    print(f"\n{r.status_code}  {url}")
    if r.status_code == 200 and r.text:
        print(r.text[:500])


404  https://www.trassenfinder.de/api/v5/infrastrukturen

404  https://www.trassenfinder.de/api/v5/version

404  https://www.trassenfinder.de/api/v5/stammdaten

404  https://www.trassenfinder.de/api/v4/infrastrukturen

404  https://www.trassenfinder.de/api/v4/version

404  https://trassenfinder.de/api/v5/infrastrukturen

404  https://trassenfinder.de/api/v5/version
